# TrafficVision-AI :: Notebook 04 — Explainable AI

---
**Version:** 2.0.0 | **Scope:** Grad-CAM, SHAP, Model Interpretability, Regulatory Compliance

---

## Table of Contents
1. [Why Explainability Matters](#1-why-explainability)
2. [Grad-CAM Implementation](#2-grad-cam)
3. [SHAP Feature Attribution](#3-shap)
4. [Per-Coordinate Analysis](#4-per-coordinate)
5. [Failure Mode Analysis](#5-failure-modes)
6. [Regulatory Compliance (EU AI Act)](#6-regulatory)
7. [Production Explainability Pipeline](#7-production)

## 1. Why Explainability Matters

For autonomous driving and safety-critical vision systems:

| Stakeholder | Explainability Need |
|-------------|--------------------|
| Safety Engineers | Understand where model looks — verify it uses road signs, not artifacts |
| Regulators (EU AI Act) | Document decision rationale for high-risk AI systems |
| MLOps Teams | Debug failures — wrong attention = data leakage or distribution shift |
| Product Teams | Build user trust by surfacing model confidence |
| Auditors | Provide post-hoc explanations for incident review |

### XAI Methods Implemented

```
Grad-CAM (Gradient-weighted Class Activation Mapping)
├── Fast (single forward + backward pass)
├── Resolution: same as last conv layer (7×7 → upsampled to 224×224)
├── Per output coordinate (x_min, y_min, x_max, y_max separately)
└── Best for: understanding spatial attention

SHAP DeepExplainer
├── Theoretically grounded (Shapley values from game theory)
├── Pixel-level attribution
├── Requires background dataset (50–200 images)
└── Best for: feature importance + regulatory documentation
```

## 2. Grad-CAM Implementation

```
GRAD-CAM COMPUTATION FLOW
==========================

Input Image (1, 224, 224, 3)
           │
           ▼
    Forward Pass
    ResNet50 Backbone
           │
           ├──────────────────────────────────┐
           │                                  │
    conv5_block3_out                    bbox_output
    (1, 7, 7, 2048)                    (1, 4)
                                              │
                                    select coord_idx=0
                                    (x_min prediction)
                                              │
           ┌──────────────────────────────────┘
           │     Backward Pass
           ▼
    ∂(x_min) / ∂(conv5_block3_out)
    Gradient tensor: (1, 7, 7, 2048)
           │
           ▼
    GlobalAvgPool over spatial dims
    Importance weights: (2048,)
           │
           ▼
    Weighted sum of feature maps
    Heatmap: (7, 7)
           │
           ▼
    ReLU (keep only positive activations)
           │
           ▼
    Bilinear upsample to (224, 224)
           │
           ▼
    Jet colormap overlay on original image
```

In [ ]:
import sys, os, numpy as np
sys.path.insert(0, os.path.abspath('..'))
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Simulate Grad-CAM output for visualization demo
np.random.seed(42)

# Synthetic traffic sign image (224x224x3)
image = np.random.rand(224, 224, 3).astype(np.float32)
# Add a bright region simulating a sign
image[60:160, 80:150] = np.array([0.9, 0.2, 0.2])  # Red sign region

# Simulated Grad-CAM heatmap (7x7 → upsampled)
raw_heatmap = np.zeros((7, 7))
raw_heatmap[2:5, 3:5] = np.array([[0.8, 0.9], [0.95, 1.0], [0.7, 0.85]])

import cv2
heatmap_upsampled = cv2.resize(raw_heatmap, (224, 224))
heatmap_uint8 = np.uint8(255 * heatmap_upsampled)
heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
overlay = np.uint8(image * 255 * 0.6 + heatmap_rgb * 0.4)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image); axes[0].set_title('Original Image', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(heatmap_upsampled, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap (x_min coord)', fontweight='bold'); axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title('Superimposed Explanation', fontweight='bold'); axes[2].axis('off')
plt.suptitle('Grad-CAM: Where does the model look to predict x_min?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('High activation (red) = most important pixels for x_min prediction')

## 4. Per-Coordinate Analysis

A key insight of our explainability system: **each bounding box coordinate uses different spatial regions**.

| Coordinate | Expected Attention Region | Interpretation |
|------------|--------------------------|----------------|
| `x_min` | Left edge of object | Model looks at left boundary |
| `y_min` | Top edge of object | Model looks at top boundary |
| `x_max` | Right edge of object | Model looks at right boundary |
| `y_max` | Bottom edge of object | Model looks at bottom boundary |

If Grad-CAM shows attention on **background** rather than the sign/light:
→ Model may be exploiting background correlations (spurious features)
→ Action: add background augmentation, collect more diverse training data

## 5. Failure Mode Analysis

### Common Failure Modes

```
FAILURE MODE 1: Background Leakage
  Symptom: Grad-CAM shows high activation on road/sky, not sign
  Cause:   Training images have correlated backgrounds
  Fix:     Random background pasting augmentation

FAILURE MODE 2: Scale Sensitivity
  Symptom: IoU drops for very small/large signs
  Cause:   GAP collapses spatial information equally
  Fix:     Multi-scale feature pyramid network (FPN)

FAILURE MODE 3: Occlusion Failure
  Symptom: Bbox prediction degrades with >30% occlusion
  Cause:   Limited occluded examples in training set
  Fix:     Synthetic occlusion augmentation

FAILURE MODE 4: Low-Light Degradation
  Symptom: IoU drops significantly in night/rain images
  Cause:   Training set biased toward daylight
  Fix:     Brightness/contrast augmentation + night image collection
```

## 6. Regulatory Compliance (EU AI Act)

TrafficVision-AI is classified as **HIGH-RISK AI** under the EU AI Act (Annex III — road traffic management and autonomous vehicles).

Mandatory documentation requirements:

| Requirement | Implementation |
|-------------|----------------|
| Technical documentation | This notebook suite + architecture docs |
| Risk management system | Drift detection + quality gates |
| Data governance | ETL validation report + DVC lineage |
| Accuracy metrics | Evaluation report (IoU, MAE, precision) |
| Human oversight | Manual promotion gate to production |
| Post-market monitoring | PSI drift detection + alerting |
| Explainability | Grad-CAM + SHAP reports per inference |

## 7. Production Explainability Pipeline

```python
# Triggered on-demand or for flagged predictions
from src.ml.explainability import ExplainabilityReport

report = ExplainabilityReport(
    keras_model=model.keras_model,
    output_dir=Path('reports/explanations/2025-01-15'),
)

# Generate Grad-CAM overlays for 10 images
meta = report.generate(
    images=X_test[:10],
    predictions=model.predict(X_test[:10]),
    image_ids=[f'frame_{i:04d}' for i in range(10)],
)
print(f'Generated {len(meta["outputs"])} explanations')
```

**Production workflow**: Every prediction with confidence < 0.80 is automatically sent to the explainability pipeline and queued for human review.

---
*TrafficVision-AI Explainable AI Notebook — Enterprise R&D Documentation*